# 04 —  Radar Data Preprocessing

This notebook converts valid complex-valued drone and bird radar measurements into reproducible model-ready range–Doppler tensors.

The objectives are to:

1. Define the target classes, labels, radar parameters, and output locations.
2. Exclude irrelevant and field-of-view edge observations.
3. Preserve session, subtype, range, time, and official-split metadata.
4. Transform each complex segment into a normalised range–Doppler tensor.
5. save and validate the official training, validation, and test arrays.
6. Create balanced, nested limited-training-data subsets.
7. Save a manifest describing the experimental subsets.


## 1. Environment, Paths, and Configuration


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

RAW_DATA_PATH = Path(
    "../data/raw/data_SAAB_SIRS_77GHz_FMCW.npy"
)

PROCESSED_DATA_DIR = Path(
    "../data/processed/official_split"
)

PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {RAW_DATA_PATH.resolve()}"
    )

print("Raw dataset:", RAW_DATA_PATH.resolve())
print("Output directory:", PROCESSED_DATA_DIR.resolve())

In [ ]:
DRONE_LABELS = {"D1", "D2", "D3", "D4", "D5", "D6"}

BIRD_LABELS = {
    "seagull",
    "pigeon",
    "raven",
    "black-headed gull",
    "seagull and black-headed gull",
    "heron"
}

LABEL_ENCODING = {
    "bird": 0,
    "drone": 1
}

SPLIT_NAMES = {
    1: "train",
    2: "validation",
    3: "test"
}

PRF_HZ = 17_000
CARRIER_FREQUENCY_HZ = 77e9
SPEED_OF_LIGHT_M_S = 299_792_458

WAVELENGTH_M = (
    SPEED_OF_LIGHT_M_S / CARRIER_FREQUENCY_HZ
)

AZIMUTH_START = 54
AZIMUTH_END = 204

MIN_DB = -60.0
MAX_DB = 0.0

RANDOM_SEED = 42

### Configuration

The six drone models D1–D6 are merged into the **drone** class, and all bird categories are merged into **bird**. Humans and corner-reflector measurements are excluded.

| Target group | Encoded label |
|---|---:|
| Bird | 0 |
| Drone | 1 |

Central slow-time indices 54–203 are retained, producing 150 Doppler samples. Relative magnitude is clipped to **[−60, 0] dB** and scaled to **[0, 1]**. Random seed 42 is used for reproducibility.


## 2. Label Mapping and Signal-Processing Utilities


In [ ]:
def extract_label(value):
    """Extract a clean target label."""
    label_array = np.asarray(value).reshape(-1)

    if label_array.size == 0:
        return "unknown"

    return str(label_array[0]).strip()


def map_target_group(label):
    """Map an original label to bird, drone, or excluded."""
    if label in DRONE_LABELS:
        return "drone"

    if label in BIRD_LABELS:
        return "bird"

    return "excluded"


def compute_range_doppler(segment):
    """
    Convert one complex radar segment from shape (5, 256)
    to a normalised float32 range–Doppler tensor (5, 150).
    """
    segment = np.asarray(segment)

    if segment.shape != (5, 256):
        raise ValueError(
            f"Expected shape (5, 256), received {segment.shape}"
        )

    central_samples = segment[
        :,
        AZIMUTH_START:AZIMUTH_END
    ]

    window = np.hanning(
        central_samples.shape[1]
    )

    windowed_signal = (
        central_samples * window[np.newaxis, :]
    )

    doppler_complex = np.fft.fftshift(
        np.fft.fft(windowed_signal, axis=1),
        axes=1
    )

    magnitude_db = 20 * np.log10(
        np.abs(doppler_complex) + 1e-12
    )

    # Express every patch relative to its maximum.
    relative_db = magnitude_db - np.max(magnitude_db)

    # Use a fixed dynamic range.
    clipped_db = np.clip(
        relative_db,
        MIN_DB,
        MAX_DB
    )

    # Scale [-60, 0] dB to [0, 1].
    normalised = (
        (clipped_db - MIN_DB)
        / (MAX_DB - MIN_DB)
    )

    return normalised.astype(np.float32)

The processing function reshapes a complex segment to **(5, 256)**, retains the central interval, applies a Hann window, computes an FFT along slow time, expresses magnitude relative to the patch maximum, clips its dynamic range, and returns a float32 tensor.

The resulting representation is a short range–Doppler patch of shape **(5, 150)**. Because it covers approximately 15 ms, it is not a conventional long-duration micro-Doppler spectrogram.


## 3. Source Dataset Loading


In [ ]:
data = np.load(
    RAW_DATA_PATH,
    allow_pickle=True
)

print("Original dataset shape:", data.shape)
print("Original dataset dtype:", data.dtype)

The source NumPy object has shape **(130, 6)**. Its rows represent measurement sessions, and its columns contain the target label, complex radar segments, range, time, official split indicator, and field-of-view edge flag.


## 4. Valid Samples in the Official Partitions


In [ ]:
valid_counts = {
    "train": 0,
    "validation": 0,
    "test": 0
}

class_counts = {
    "train": {"bird": 0, "drone": 0},
    "validation": {"bird": 0, "drone": 0},
    "test": {"bird": 0, "drone": 0}
}

for row in data:
    label = extract_label(row[0])
    target_group = map_target_group(label)

    if target_group == "excluded":
        continue

    splits = np.asarray(row[4]).reshape(-1)
    edge_flags = np.asarray(row[5]).reshape(-1)

    valid_mask = edge_flags == 0

    for split_value, split_name in SPLIT_NAMES.items():
        count = int(
            np.sum(valid_mask & (splits == split_value))
        )

        valid_counts[split_name] += count
        class_counts[split_name][target_group] += count

print("Valid samples per split:")
print(json.dumps(valid_counts, indent=2))

print("\nValid class counts per split:")
print(json.dumps(class_counts, indent=2))

In [ ]:
split_count_records = []

for split_name, groups in class_counts.items():
    for target_group, count in groups.items():
        split_count_records.append({
            "split": split_name,
            "target_group": target_group,
            "samples": count
        })

split_counts_df = pd.DataFrame(
    split_count_records
)

split_counts_table = split_counts_df.pivot(
    index="split",
    columns="target_group",
    values="samples"
)

split_counts_table["total"] = (
    split_counts_table.sum(axis=1)
)

display(split_counts_table)

### Official-Partition Counts

After excluding humans, corner-reflector measurements, and 67 field-of-view edge samples, **66,493 valid drone and bird segments** remain.

| Split | Bird | Drone | Total |
|---|---:|---:|---:|
| Training | 5,749 | 46,760 | 52,509 |
| Validation | 990 | 5,998 | 6,988 |
| Test | 997 | 5,999 | 6,996 |
| **Total** | **7,736** | **58,757** | **66,493** |

The drone-to-bird ratios are approximately 8.13:1 in training, 6.06:1 in validation, and 6.02:1 in testing.

The validation and test distributions are preserved for evaluation. Class imbalance will be handled during training through balanced subsets, class weighting, or balanced sampling rather than by modifying evaluation data.


### Official-Split Limitation

The provided split is segment-level: temporally related segments from the same recording session may occur in training, validation, and testing. It is retained for comparability with the associated study, but later modelling will also require a session-independent protocol to evaluate generalisation to unseen sessions.


## 5. Batched Range–Doppler Tensor Generation


In [ ]:
def transform_segment_batch(segment_batch):
    """
    Transform a batch from (N, 5, 256) complex segments
    into (N, 5, 150) normalised range–Doppler tensors.
    """
    central_samples = segment_batch[
        :,
        :,
        AZIMUTH_START:AZIMUTH_END
    ]

    window = np.hanning(
        central_samples.shape[2]
    ).astype(np.float32)

    windowed = (
        central_samples
        * window[np.newaxis, np.newaxis, :]
    )

    doppler_complex = np.fft.fftshift(
        np.fft.fft(windowed, axis=2),
        axes=2
    )

    magnitude_db = 20 * np.log10(
        np.abs(doppler_complex) + 1e-12
    )

    patch_maximum = np.max(
        magnitude_db,
        axis=(1, 2),
        keepdims=True
    )

    relative_db = magnitude_db - patch_maximum

    clipped_db = np.clip(
        relative_db,
        MIN_DB,
        MAX_DB
    )

    normalised = (
        (clipped_db - MIN_DB)
        / (MAX_DB - MIN_DB)
    )

    return normalised.astype(np.float32)

In [ ]:
def process_official_split(
    dataset,
    split_value,
    split_name,
    expected_count
):
    """
    Process and save one official dataset partition.
    """
    X = np.empty(
        (expected_count, 5, 150),
        dtype=np.float32
    )

    y = np.empty(
        expected_count,
        dtype=np.uint8
    )

    metadata_records = []
    output_position = 0

    for session_id, row in enumerate(dataset):
        original_label = extract_label(row[0])
        target_group = map_target_group(original_label)

        if target_group == "excluded":
            continue

        segments = np.asarray(row[1])
        ranges = np.asarray(row[2]).reshape(-1)
        times = np.asarray(row[3]).reshape(-1)
        splits = np.asarray(row[4]).reshape(-1)
        edge_flags = np.asarray(row[5]).reshape(-1)

        selected_indices = np.where(
            (splits == split_value)
            & (edge_flags == 0)
        )[0]

        if selected_indices.size == 0:
            continue

        # Each selected column is one flattened segment.
        segment_batch = (
            segments[:, selected_indices]
            .T
            .reshape(-1, 5, 256)
        )

        transformed_batch = transform_segment_batch(
            segment_batch
        )

        batch_size = len(selected_indices)
        batch_end = output_position + batch_size

        X[output_position:batch_end] = (
            transformed_batch
        )

        y[output_position:batch_end] = (
            LABEL_ENCODING[target_group]
        )

        for local_position, segment_id in enumerate(
            selected_indices
        ):
            metadata_records.append({
                "array_index": output_position + local_position,
                "session_id": session_id,
                "segment_id": int(segment_id),
                "original_label": original_label,
                "target_group": target_group,
                "encoded_label": LABEL_ENCODING[
                    target_group
                ],
                "range_m": float(ranges[segment_id]),
                "time_s": float(times[segment_id]),
                "official_split": split_name,
                "edge_flag": int(edge_flags[segment_id])
            })

        output_position = batch_end

    if output_position != expected_count:
        raise RuntimeError(
            f"{split_name}: expected {expected_count} samples, "
            f"but processed {output_position}"
        )

    metadata_df = pd.DataFrame(metadata_records)

    if not np.isfinite(X).all():
        raise ValueError(
            f"{split_name}: processed tensors contain "
            "NaN or infinite values"
        )

    np.save(
        PROCESSED_DATA_DIR / f"X_{split_name}.npy",
        X
    )

    np.save(
        PROCESSED_DATA_DIR / f"y_{split_name}.npy",
        y
    )

    metadata_df.to_csv(
        PROCESSED_DATA_DIR / f"metadata_{split_name}.csv",
        index=False
    )

    print(f"{split_name}:")
    print("  X shape:", X.shape)
    print("  y shape:", y.shape)
    print("  Metadata rows:", len(metadata_df))
    print("  Minimum tensor value:", X.min())
    print("  Maximum tensor value:", X.max())
    print("  Bird samples:", np.sum(y == 0))
    print("  Drone samples:", np.sum(y == 1))

    return X, y, metadata_df

The batched implementation processes all selected segments from one session together. Preallocation guarantees the expected sample counts and avoids repeatedly expanding large Python lists.


### 5.1 Training Partition


In [ ]:
X_train, y_train, metadata_train = (
    process_official_split(
        dataset=data,
        split_value=1,
        split_name="train",
        expected_count=valid_counts["train"]
    )
)

In [ ]:
import gc

del X_train, y_train, metadata_train
gc.collect()

The training partition was successfully converted to a tensor array of shape **(52,509, 5, 150)** with 52,509 aligned labels and metadata records. It contains 5,749 birds and 46,760 drones.

All values are finite and lie in **[0, 1]**. The arrays were released from memory after saving.


### 5.2 Validation and Test Partitions


In [ ]:
for split_value, split_name in [
    (2, "validation"),
    (3, "test")
]:
    X_split, y_split, metadata_split = (
        process_official_split(
            dataset=data,
            split_value=split_value,
            split_name=split_name,
            expected_count=valid_counts[split_name]
        )
    )

    del X_split, y_split, metadata_split
    gc.collect()

The validation tensor has shape **(6,988, 5, 150)** and contains 990 birds and 5,998 drones. The test tensor has shape **(6,996, 5, 150)** and contains 997 birds and 5,999 drones.

No samples were lost unexpectedly. Tensor, label, and metadata counts match the precomputed official-partition totals.


## 6. Saved-File and Metadata Integrity


In [ ]:
expected_files = [
    "X_train.npy",
    "y_train.npy",
    "metadata_train.csv",
    "X_validation.npy",
    "y_validation.npy",
    "metadata_validation.csv",
    "X_test.npy",
    "y_test.npy",
    "metadata_test.csv"
]

file_records = []

for filename in expected_files:
    path = PROCESSED_DATA_DIR / filename

    file_records.append({
        "filename": filename,
        "exists": path.exists(),
        "size_mb": (
            path.stat().st_size / (1024 ** 2)
            if path.exists()
            else np.nan
        )
    })

file_verification_df = pd.DataFrame(file_records)

display(
    file_verification_df.style.format({
        "size_mb": "{:.2f}"
    })
)

All nine expected files were created successfully:

| File | Approximate size |
|---|---:|
| X_train.npy | 150.23 MB |
| y_train.npy | 0.05 MB |
| metadata_train.csv | 3.50 MB |
| X_validation.npy | 19.99 MB |
| y_validation.npy | 0.01 MB |
| metadata_validation.csv | 0.49 MB |
| X_test.npy | 20.02 MB |
| y_test.npy | 0.01 MB |
| metadata_test.csv | 0.46 MB |

Their combined size is approximately **194.76 MB**.


In [ ]:
X_train_check = np.load(
    PROCESSED_DATA_DIR / "X_train.npy",
    mmap_mode="r"
)

y_train_check = np.load(
    PROCESSED_DATA_DIR / "y_train.npy",
    mmap_mode="r"
)

metadata_train_check = pd.read_csv(
    PROCESSED_DATA_DIR / "metadata_train.csv"
)

print("Tensor shape:", X_train_check.shape)
print("Tensor dtype:", X_train_check.dtype)
print("Label shape:", y_train_check.shape)
print("Label dtype:", y_train_check.dtype)
print("Metadata shape:", metadata_train_check.shape)
print("All values finite:", np.isfinite(X_train_check).all())
print(
    "Labels match metadata:",
    np.array_equal(
        np.asarray(y_train_check),
        metadata_train_check[
            "encoded_label"
        ].to_numpy()
    )
)

### Reloading and Consistency Results

Memory-mapped reloading confirmed:

- Training tensor shape: **(52,509, 5, 150)**
- Tensor type: **float32**
- Label shape: **(52,509,)**
- Label type: **uint8**
- Metadata shape: **(52,509, 10)**
- All tensor values are finite
- Array labels exactly match metadata labels

The ten metadata columns preserve array index, session, segment, original label, broad group, encoded label, range, time, official split, and edge status.


## 7. Visual Quality Control


In [ ]:
bird_index = int(
    np.where(np.asarray(y_train_check) == 0)[0][0]
)

drone_index = int(
    np.where(np.asarray(y_train_check) == 1)[0][0]
)

fig, axes = plt.subplots(
    2,
    1,
    figsize=(12, 6),
    constrained_layout=True
)

for axis, sample_index, title in [
    (axes[0], bird_index, "Processed Bird Example"),
    (axes[1], drone_index, "Processed Drone Example")
]:
    image = axis.imshow(
        X_train_check[sample_index],
        aspect="auto",
        origin="lower",
        cmap="magma",
        vmin=0,
        vmax=1
    )

    metadata_row = metadata_train_check.iloc[
        sample_index
    ]

    axis.set_title(
        f"{title} — {metadata_row['original_label']} "
        f"at {metadata_row['range_m']:.1f} m"
    )
    axis.set_xlabel("Doppler bin")
    axis.set_ylabel("Range cell")
    axis.set_yticks(range(5))

fig.colorbar(
    image,
    ax=axes,
    label="Normalised magnitude",
    shrink=0.85
)

plt.show()

### Visual Validation Result

One processed seagull and one processed D1 segment preserve the expected **(5, 150)** structure and central-range-cell energy concentration. Both use identical normalisation and colour limits.

The seagull example contains a dominant component slightly away from the centre Doppler bin, while D1 has a stronger near-centre response. These observations confirm successful processing but do not independently establish class separability.

The official arrays pass structural, numerical, metadata-alignment, and visual-quality checks.


## 8. Balanced Limited-Training-Data Subsets


The minority class contains 5,749 training birds. The largest balanced subset therefore contains all 5,749 birds and 5,749 randomly selected drones.

The smaller subsets are defined relative to this per-class maximum. Only indices into the base tensor are saved, avoiding duplication of the 150 MB training file.


In [ ]:
from math import ceil

LIMITED_DATA_DIR = Path(
    "../data/processed/limited_subsets"
)

LIMITED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

y_train_base = np.load(
    PROCESSED_DATA_DIR / "y_train.npy",
    mmap_mode="r"
)

metadata_train_base = pd.read_csv(
    PROCESSED_DATA_DIR / "metadata_train.csv"
)

bird_indices = np.where(
    np.asarray(y_train_base) == 0
)[0]

drone_indices = np.where(
    np.asarray(y_train_base) == 1
)[0]

minority_count = len(bird_indices)

print("Available birds:", len(bird_indices))
print("Available drones:", len(drone_indices))
print("Maximum balanced samples per class:", minority_count)

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)

shuffled_bird_indices = rng.permutation(
    bird_indices
)

shuffled_drone_indices = rng.permutation(
    drone_indices
)[:minority_count]

subset_fractions = {
    "10_percent": 0.10,
    "25_percent": 0.25,
    "50_percent": 0.50,
    "100_percent": 1.00
}

subset_summary_records = []
saved_subset_indices = {}

for subset_name, fraction in subset_fractions.items():
    samples_per_class = ceil(
        minority_count * fraction
    )

    selected_birds = shuffled_bird_indices[
        :samples_per_class
    ]

    selected_drones = shuffled_drone_indices[
        :samples_per_class
    ]

    selected_indices = np.concatenate([
        selected_birds,
        selected_drones
    ])

    # Randomise order without changing membership.
    subset_rng = np.random.default_rng(
        RANDOM_SEED
    )
    selected_indices = subset_rng.permutation(
        selected_indices
    )

    selected_labels = np.asarray(
        y_train_base[selected_indices]
    )

    np.save(
        LIMITED_DATA_DIR / f"indices_{subset_name}.npy",
        selected_indices.astype(np.int64)
    )

    subset_metadata = (
        metadata_train_base
        .iloc[selected_indices]
        .copy()
    )

    subset_metadata.insert(
        0,
        "subset_position",
        np.arange(len(subset_metadata))
    )

    subset_metadata.to_csv(
        LIMITED_DATA_DIR / f"metadata_{subset_name}.csv",
        index=False
    )

    saved_subset_indices[subset_name] = set(
        selected_indices.tolist()
    )

    subset_summary_records.append({
        "subset": subset_name,
        "fraction": fraction,
        "samples_per_class": samples_per_class,
        "bird_samples": int(
            np.sum(selected_labels == 0)
        ),
        "drone_samples": int(
            np.sum(selected_labels == 1)
        ),
        "total_samples": len(selected_indices),
        "represented_sessions": int(
            subset_metadata["session_id"].nunique()
        )
    })

subset_summary_df = pd.DataFrame(
    subset_summary_records
)

display(subset_summary_df)

### Limited-Subset Results

| Subset | Fraction | Samples per class | Birds | Drones | Total | Sessions represented |
|---|---:|---:|---:|---:|---:|---:|
| 10% | 0.10 | 575 | 575 | 575 | 1,150 | 97 |
| 25% | 0.25 | 1,438 | 1,438 | 1,438 | 2,876 | 99 |
| 50% | 0.50 | 2,875 | 2,875 | 2,875 | 5,750 | 100 |
| 100% | 1.00 | 5,749 | 5,749 | 5,749 | 11,498 | 100 |

Here, 100% means the largest **balanced** real-data subset supported by the minority class; it does not mean all 52,509 official training observations. The complete imbalanced training partition will be evaluated separately.


### 8.1 Nested-Subset Validation


In [ ]:
subset_order = [
    "10_percent",
    "25_percent",
    "50_percent",
    "100_percent"
]

for smaller, larger in zip(
    subset_order[:-1],
    subset_order[1:]
):
    is_nested = saved_subset_indices[
        smaller
    ].issubset(
        saved_subset_indices[larger]
    )

    print(
        f"{smaller} contained in {larger}:",
        is_nested
    )

for subset_name in subset_order:
    indices = np.load(
        LIMITED_DATA_DIR
        / f"indices_{subset_name}.npy"
    )

    assert len(indices) == len(np.unique(indices))
    assert indices.min() >= 0
    assert indices.max() < len(y_train_base)

print("All subset indices are unique and valid.")

The validation confirms that 10% is contained in 25%, 25% in 50%, and 50% in 100%. Every index is unique, non-negative, and within the official training-array boundaries.

This design improves comparability: increasing the data fraction adds samples without replacing those used in smaller subsets.


### 8.2 Session-Coverage Limitation

The 10% subset already represents 97 of the 100 available drone/bird sessions; 25% represents 99, and the larger subsets represent all 100.

These subsets therefore simulate **limited labelled segments**, not limited recording sessions. They are appropriate for the primary synthetic-augmentation experiment, but a separate session-independent experiment is necessary for unseen-session generalisation.


## 9. Experiment Manifest


In [ ]:
subset_manifest = {
    "random_seed": RANDOM_SEED,
    "label_encoding": LABEL_ENCODING,
    "base_training_samples": int(len(y_train_base)),
    "available_bird_samples": int(len(bird_indices)),
    "available_drone_samples": int(len(drone_indices)),
    "balanced_maximum_per_class": int(minority_count),
    "subsets": {}
}

for record in subset_summary_records:
    subset_manifest["subsets"][record["subset"]] = {
        "fraction": float(record["fraction"]),
        "samples_per_class": int(
            record["samples_per_class"]
        ),
        "bird_samples": int(record["bird_samples"]),
        "drone_samples": int(record["drone_samples"]),
        "total_samples": int(record["total_samples"]),
        "represented_sessions": int(
            record["represented_sessions"]
        )
    }

manifest_path = (
    LIMITED_DATA_DIR / "subset_manifest.json"
)

with open(manifest_path, "w", encoding="utf-8") as file:
    json.dump(
        subset_manifest,
        file,
        indent=2
    )

print("Manifest saved:", manifest_path.resolve())

The manifest records the random seed, label encoding, base training size, available class counts, balanced maximum, subset fractions, sample counts, and session coverage. It allows subsequent notebooks to load the experimental design without redefining it manually.


## 10. Final Preprocessing Conclusion

The preprocessing pipeline successfully produced:

- 52,509 official training tensors
- 6,988 official validation tensors
- 6,996 official test tensors
- Float32 range–Doppler inputs of shape **(5, 150)**
- Values normalised to **[0, 1]**
- Binary uint8 labels
- Complete sample-level metadata
- Four balanced, nested limited-data subsets
- A reproducible subset manifest

Two real-only training references will be evaluated:

1. The complete imbalanced official training set with 52,509 samples
2. The balanced 100% subset with 11,498 samples

The 10%, 25%, and 50% balanced subsets will quantify performance under limited labelled-data conditions. Synthetic samples will later be added only to these training subsets. Validation and testing will remain entirely real.

The main comparison is

$$
\text{limited real training} \quad \text{versus} \quad \text{the same limited real training plus synthetic augmentation}.
$$


The data are ready for baseline CNN development in **05_baseline_classification.ipynb**.
